# Phase 2 — Clustering & Prototype Discovery (Days 6–10)

L2-normalizes the UNSAFE embeddings, sweeps K-means over `k`, selects `k*` by
silhouette, and writes the prototype taxonomy. **CPU-only — no GPU needed.**

### Step 0 — get the repo onto this runtime

In [ ]:
import os, subprocess
from pathlib import Path

# ── EDIT THIS if you have a GitHub remote ────────────────────────────────────
GITHUB_URL = "https://github.com/yogijoshi86/SLMProject.git"
# ─────────────────────────────────────────────────────────────────────────────

TARGET = Path("/content/SLMProject")

if TARGET.is_dir() and (TARGET / "src" / "guardrail_audit").is_dir():
    print("Repo already present — pulling latest…")
    subprocess.run(["git", "-C", str(TARGET), "pull", "--ff-only"], check=True)

elif GITHUB_URL:
    print("Cloning from GitHub…")
    subprocess.run(["git", "clone", GITHUB_URL, str(TARGET)], check=True)
    print("Cloned to", TARGET)

else:
    # ── Google Drive fallback ─────────────────────────────────────────────────
    # Mount Drive once then point DRIVE_PATH at wherever you stored the folder.
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_PATH = "/content/drive/MyDrive/SLMProject"   # adjust if needed
    if not Path(DRIVE_PATH).is_dir():
        raise FileNotFoundError(
            f"Could not find the repo at {DRIVE_PATH}. "
            "Either set GITHUB_URL above, or copy the SLMProject folder to your Drive "
            "and update DRIVE_PATH."
        )
    import shutil
    shutil.copytree(DRIVE_PATH, str(TARGET))
    print("Copied from Drive to", TARGET)

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path("/content/SLMProject")
assert (REPO_ROOT / "src" / "guardrail_audit").is_dir(), \
    f"src/guardrail_audit not found under {REPO_ROOT}. Did the previous cell succeed?"

sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)
print("Repo root:", REPO_ROOT)

In [ ]:
# Pin exact versions proven compatible: transformers 4.44.2 + accelerate 0.33.0 + bitsandbytes 0.45.x
%pip install -q -e ".[quant,explainer,dev]" \
    "torch>=2.4.0" "torchvision>=0.19.0" \
    "transformers==4.44.2" \
    "accelerate==0.33.0" \
    "bitsandbytes>=0.45.0"

In [ ]:
# Restart the runtime so the upgraded transformers is loaded fresh.
# After restart, re-run from the HF auth cell downwards (CLONE/LOCATE/INSTALL are done).
import os
print("Restarting runtime to pick up upgraded packages...")
os.kill(os.getpid(), 9)

In [ ]:
from guardrail_audit.utils import load_config, set_seed

# colab_smoke.yaml = 500 prompts + int8 (fits a free T4). Swap to default.yaml for full runs.
CONFIG = "config/colab_smoke.yaml"
cfg = load_config(CONFIG)
set_seed(cfg.seed)
cfg

### Build prototypes (sweep + silhouette + exemplars)

In [ ]:
from guardrail_audit.clustering import build_prototypes

taxonomy = build_prototypes(
    data_path=cfg.paths.embeddings,
    taxonomy_path=cfg.paths.taxonomy,
    k_min=cfg.clustering.k_min,
    k_max=cfg.clustering.k_max,
    n_init=cfg.clustering.n_init,
    seed=cfg.seed,
    top_exemplars=cfg.clustering.top_exemplars,
    k_cap=cfg.clustering.k_cap,
)
print("best k*:", taxonomy["meta"]["best_k"], "| silhouette:", round(taxonomy["meta"]["best_silhouette"], 4))

### Plot the silhouette / inertia sweep

In [ ]:
import matplotlib.pyplot as plt

sweep = taxonomy["meta"]["sweep"]
ks = [s["k"] for s in sweep]
sil = [s["silhouette"] for s in sweep]
inertia = [s["inertia"] for s in sweep]

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(ks, sil, "o-", color="tab:blue", label="silhouette")
ax1.axhline(0.45, ls="--", color="tab:blue", alpha=0.5, label="H2 target 0.45")
ax1.set_xlabel("k"); ax1.set_ylabel("silhouette", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(ks, inertia, "s--", color="tab:red", alpha=0.6)
ax2.set_ylabel("inertia", color="tab:red")
ax1.axvline(taxonomy["meta"]["best_k"], color="green", alpha=0.4)
plt.title("K-means sweep"); ax1.legend(loc="upper right"); plt.show()

### Inspect each prototype's exemplars

In [ ]:
for key, proto in taxonomy["prototypes"].items():
    print(f"\n=== {key}  (size={proto['cluster_size']}, cats={proto['dominant_categories'][:3]}) ===")
    for i, ex in enumerate(proto["top_exemplars"][:3], 1):
        print(f"  {i}. {ex[:140]}")

### Day 10 — manual thematic review (do this by hand)

The taxonomy JSON has `"label": "TODO"` and `"failure_mode": "TODO"` per prototype.
Read the exemplars above and edit `artifacts/prototypes_taxonomy_smoke.json`
to give each cluster a human name (e.g. *"Obfuscated Hate Speech via Homoglyphs"*).
The explainer in Phase 3 surfaces these labels.

In [ ]:
# Optional helper: fill labels here, then re-save.
import json
with open(cfg.paths.taxonomy) as f:
    tax = json.load(f)

# Example — edit these:
# tax["prototypes"]["prototype_0"]["label"] = "Roleplay / Hypothetical Framing"
# tax["prototypes"]["prototype_0"]["failure_mode"] = "guard over-triggers on fictional framing"

with open(cfg.paths.taxonomy, "w") as f:
    json.dump(tax, f, indent=2)
print("Saved.")

**Next:** open `03_audit.ipynb`.